# Session Memory Compaction

Claude와의 장기 대화는 context 제한을 초과하여 중요한 정보를 잃을 수 있습니다. coding assistant, 창작 도구, 고객 서비스 agent 중 무엇을 만들든 session memory 관리는 연속성과 품질을 유지하는 데 중요합니다.

이 cookbook은 불편한 context 제한 중단을 피하기 위해 **session memory를 선제적으로 관리**하는 방법을 가르칩니다. context가 가득 찰 때까지 기다리는 사후 대응 방식과 달리, 필요할 때 즉시 compaction할 수 있도록 백그라운드에서 session memory를 구축하는 방법을 배웁니다.

**관련:** agentic workflow에서 SDK 기반 자동 compaction은 [Automatic Context Compaction](../tool_use/automatic-context-compaction.ipynb)을 참조하세요. 이 cookbook은 대화형 application을 위한 수동 제어 패턴에 집중합니다.

## 학습 목표

이 cookbook을 마치면 다음을 할 수 있습니다.

- compaction 사이에 중요한 context를 보존하는 효과적인 session memory 프롬프트 작성
- background threading으로 **즉시 compaction**을 구현하여 사용자 대기 시간 제거
- prompt caching을 적용하여 백그라운드 memory 갱신 비용 약 80% 절감
- 사용 사례에 따라 적절한 compaction 전략(기존 방식 대 즉시 방식) 선택

## 사전 요구 사항 및 설정

이 가이드를 따라 하기 전에 다음을 갖추었는지 확인하세요.

**필요한 지식**
- Claude API 사용과 message 형식에 대한 기본 이해
- Python threading 개념에 대한 지식(도움이 되지만 필수는 아님)

**필요한 도구**
- Python 3.11 이상
- Anthropic API key
- Anthropic SDK

### 설치

먼저 필요한 dependency를 설치합니다.

In [ ]:
%%capture
%pip install -U anthropic python-dotenv

In [1]:
import anthropic
from anthropic.types import MessageParam, TextBlockParam
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"

In [3]:
# Helper 함수
def truncate_response(text: str, max_lines: int = 15) -> str:
    """출력을 더 깔끔하게 표시하도록 긴 응답을 자릅니다."""
    lines = text.strip().split("\n")
    if len(lines) <= max_lines:
        return text
    return "\n".join(lines[:max_lines]) + f"\n... ({len(lines) - max_lines} more lines)"


def remove_thinking_blocks(text: str) -> tuple[str, str]:
    """텍스트에서 <think>...</think> block을 제거합니다."""
    import re

    matches = re.findall(r"<think>.*?</think>", text, flags=re.DOTALL)
    cleaned = re.sub(r"<think>.*?</think>\s*", "", text, flags=re.DOTALL).strip()
    return cleaned, "".join(matches)


def add_cache_control(messages: list[dict]) -> list[MessageParam]:
    """prompt caching을 위해 마지막 user message에 cache_control을 추가합니다.

    prompt caching이 작동하려면 요청 간 message prefix 구조가 같아야 합니다.
    일관성을 위해 모든 message를 list 형식으로 변환하고, 표준 API 호출 패턴에 맞도록
    마지막 user message에 cache_control을 배치합니다.
    """
    cached_messages: list[MessageParam] = []
    last_user_idx = None

    # 마지막 user message index 찾기
    for i, msg in enumerate(messages):
        if msg["role"] == "user":
            last_user_idx = i

    for i, msg in enumerate(messages):
        content = msg["content"]
        text = content if isinstance(content, str) else content[0]["text"]

        content_block: TextBlockParam = {"type": "text", "text": text}
        if i == last_user_idx:
            content_block["cache_control"] = {"type": "ephemeral"}

        cached_messages.append({"role": msg["role"], "content": [content_block]})

    return cached_messages


def estimate_tokens(text: str) -> int:
    """기초적인 token 추정: 4문자당 1 token."""
    return len(text) // 4

/root/.pyenv/versions/3.13.11/lib/python3.13/site-packages/coconut/compiler/util.py:676: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  return Regex(regex, options)
/root/.pyenv/versions/3.13.11/lib/python3.13/site-packages/coconut/compiler/util.py:457: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  result = add_action(grammar, unpack).parseWithTabs().transformString(text)


In [4]:
SESSION_MEMORY_PROMPT = """
Compress the conversation into a structured summary
that preserves all information needed to continue work seamlessly. Optimize for the assistant's
ability to continue working, not human readability.

<analysis-instructions>
Before generating your summary, analyze the transcript in <think>...</think> tags:
1. What did the user originally request? (Exact phrasing)
2. What actions succeeded? What failed and why?
3. Did the user correct or redirect the assistant at any point?
4. What was actively being worked on at the end?
5. What tasks remain incomplete or pending?
6. What specific details (IDs, paths, values, names) must survive compression?
</analysis-instructions>

<summary-format>
## User Intent
The user's original request and any refinements. Use direct quotes for key requirements.
If the user's goal evolved during the conversation, capture that progression.

## Completed Work
Actions successfully performed. Be specific:
- What was created, modified, or deleted
- Exact identifiers (file paths, record IDs, URLs, names)
- Specific values, configurations, or settings applied

## Errors & Corrections
- Problems encountered and how they were resolved
- Approaches that failed (so they aren't retried)
- User corrections: "don't do X", "actually I meant Y", "that's wrong because..."
Capture corrections verbatim—these represent learned preferences.

## Active Work
What was in progress when the session ended. Include:
- The specific task being performed
- Direct quotes showing exactly where work left off
- Any partial results or intermediate state

## Pending Tasks
Remaining items the user requested that haven't been started.
Distinguish between "explicitly requested" and "implied/assumed."

## Key References
Important details needed to continue:
- Identifiers: IDs, paths, URLs, names, keys
- Values: numbers, dates, configurations, credentials (redacted)
- Context: relevant background information, constraints, preferences
- Citations: sources referenced during the conversation
</summary-format>

<preserve-rules>
Always preserve when present:
- Exact identifiers (IDs, paths, URLs, keys, names)
- Error messages verbatim
- User corrections and negative feedback
- Specific values, formulas, or configurations
- Technical constraints or requirements discovered
- The precise state of any in-progress work
</preserve-rules>

<compression-rules>
- Weight recent messages more heavily—the end of the transcript is the active context
- Omit pleasantries, acknowledgments, and filler ("Sure!", "Great question")
- Omit system context that will be re-injected separately
- Keep each section under 500 words; condense older content to make room for recent
- If you must cut details, preserve: user corrections > errors > active work > completed work
</compression-rules>
"""

### 기존 compacting을 사용하는 코드 예제
기존 compaction에서는 token threshold에 도달하면 요약을 하나 생성합니다.
기존 compaction은 느립니다. context 제한에 도달하면 요약을 기다립니다.


```
기존 COMPACTION (느림)
─────────────────────────────
Turn 1 → Turn 2 → Turn 3 → ... → Turn N → CONTEXT 가득 참!
                                              │
                                              ▼
                                    ┌─────────────────┐
                                    │ 요약 생성│
                                    │ ( 사용자 대기! ) │
                                    └─────────────────┘
                                              │
                                              ▼
                                         계속

```

In [5]:
import time


class TraditionalCompactingChatSession:
    """사후 compaction을 사용하는 기존 chat session."""

    def __init__(self, system_message="You are a helpful assistant", context_limit: int = 10000):
        self.system_message = system_message
        self.context_limit = context_limit  # model 제한을 넘지 않도록 대화가 compact되는 지점.
        self.messages = []
        self.current_context_window_tokens = 0
        self.summary = None

    def chat(self, user_message: str) -> tuple[str, anthropic.types.Usage]:
        # 기존 compaction에서는 사용자가 message를 보낼 때 compact 필요 여부를 확인. 이상적이지 않음!
        if self.current_context_window_tokens >= self.context_limit:
            print(
                f"\n🧹 Context window at {self.current_context_window_tokens} tokens. Limit exceeded, compacting session memory..."
            )
            self.compact()  # 새 user message 이전의 모든 내용 compact

        self.messages.append({"role": "user", "content": user_message})
        print(f"\nUser: {user_message}")

        response = client.messages.create(
            model=MODEL,
            max_tokens=3500,
            system=self.system_message,
            messages=add_cache_control(self.messages),
        )
        assistant_message = response.content[0].text
        self.messages.append({"role": "assistant", "content": assistant_message})

        print(f"\nAssistant: \n{truncate_response(assistant_message, max_lines=15)}")

        # 다음 user message 이전 대화의 대략적인 현재 token 수
        cache_read = getattr(response.usage, "cache_read_input_tokens", 0) or 0
        total_input = response.usage.input_tokens + cache_read
        self.current_context_window_tokens = total_input + response.usage.output_tokens

        print(
            f"Input={total_input:,}, Prompt cached used= {cache_read > 0} | "
            f"Output={response.usage.output_tokens:,} | "
            f"Messages={len(self.messages)}"
        )
        return assistant_message, response.usage

    def compact(self) -> None:
        start_time = time.perf_counter()

        response = client.messages.create(
            model=MODEL,
            max_tokens=5000,
            system=self.system_message,  # cache 공유를 위해 주 대화와 동일
            messages=add_cache_control(self.messages)
            + [{"role": "user", "content": SESSION_MEMORY_PROMPT}],
        )
        elapsed = time.perf_counter() - start_time

        # 새 요약 message 생성
        self.summary, removed_text = remove_thinking_blocks(
            response.content[0].text
        )  # session memory에는 필요하지 않으므로 <think> block 제거
        approximate_summary_tokens = response.usage.output_tokens - round(
            len(removed_text) / 4
        )  # 요약에서 제거된 token의 대략적인 추정

        # 이전 message들을 새 요약 message로 교체
        self.messages = [
            {
                "role": "user",
                "content": f"""This session is being continued from a previous conversation. Here is the session memory: {self.summary}.Continue from where we left off.""",
            }
        ]

        # 방금 compact했다면 token 감소량 표시
        reduction = self.current_context_window_tokens - approximate_summary_tokens
        pct = (reduction / self.current_context_window_tokens) * 100

        print(f"\n{'-' * 60}")
        print("📝 New session memory created.")
        print(
            f"✅ Tokens reduced: {self.current_context_window_tokens:,} → {approximate_summary_tokens:.0f} ({reduction:,} tokens saved, {pct:.0f}% reduction)"
        )
        print(f"⏱️ Compaction time: {elapsed:.2f}s (user waiting...)")
        print(f" Cache used: {getattr(response.usage, 'cache_read_input_tokens', 0) > 0}")
        print(f"{'-' * 60}")

        # compact된 상태를 반영하도록 token 수 갱신
        self.current_context_window_tokens = approximate_summary_tokens

/root/.pyenv/versions/3.13.11/lib/python3.13/site-packages/coconut/compiler/util.py:403: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  grammar.streamline()
/root/.pyenv/versions/3.13.11/lib/python3.13/site-packages/coconut/compiler/util.py:457: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  result = add_action(grammar, unpack).parseWithTabs().transformString(text)


아래에서는 이야기 작성을 돕는 LLM과 작가 사이의 대화를 시뮬레이션합니다.

In [6]:
SYSTEM_PROMPT = """
You are a short story writer who helps authors develop their ideas into compelling narratives.

## What You Do

**Plot Development**
- Help authors work through story structure, pacing, and narrative arc
- Identify plot holes, inconsistencies, or missed opportunities
- Suggest ways to raise stakes, add tension, or deepen conflict
- Brainstorm twists, resolutions, and scene transitions

**Character Development**
- Develop backstories, motivations, and internal conflicts
- Ensure characters have distinct voices and consistent behavior
- Explore character relationships and how they drive the plot
- Help authors understand what their characters want vs. what they need

**Drafting**
- Write short stories or scenes based on the author's ideas and direction
- Match tone, genre conventions, and stylistic preferences
- Show rather than tell when bringing scenes to life
- Craft dialogue that reveals character and advances plot

## How You Work
- You are the lead writer. When you disagree with a creative choice, say so respectfully, but ultimately defer to what the author wants.
- DO NOT ask the user to provide more context or clarify their request. Assume you have enough information to proceed.
"""

In [7]:
session = TraditionalCompactingChatSession(system_message=SYSTEM_PROMPT)

# 시뮬레이션 대화
messages = [
    "I want to create a story about a young detective solving a mysterious case in a small town. Generate 3 well thought out plot ideas for me to consider.",
    "I don't like those ideas, can you think of one plot something more unique and unexpected?",
    "Ok I like it. Can you help me develop the main character's backstory and motivations?",
    "Can you draft a detailed outline for the story, breaking it down into chapters and key events?",
    "Can you draft me a first chapter based on the plot and character ideas we've discussed so far? Make it around 2,000 words.",
    "Can you draft a second chapter that builds on the first one, introducing a new twist in the mystery?",
]

print("Starting conversation...\n")

turn_count = 0

for _i, message in enumerate(messages, 1):
    turn_count += 1
    print(f"==============================================\nTurn {turn_count}:\n")
    response, usage = session.chat(message)

Starting conversation...

Turn 1:


User: I want to create a story about a young detective solving a mysterious case in a small town. Generate 3 well thought out plot ideas for me to consider.

Assistant: 
# Three Mystery Plot Ideas

## 1. **The Vanishing Choir**

**Setup:** In the sleepy town of Millbrook, the entire church choir—twelve people ranging from teenagers to retirees—disappears during their weekly Thursday night practice. The church was locked from the inside, their belongings left behind, including phones and car keys. No signs of struggle, no broken windows. Just an empty sanctuary and sheet music scattered across the floor.

**The Twist:** Your young detective discovers the choir members didn't disappear—they're hiding. Twenty years ago, they witnessed the town's beloved mayor commit a hit-and-run that killed a drifter. They stayed silent, bound by threats and their own complicity. Now the mayor is dying and has hired someone to ensure his secret dies with him. The choir

여러 turn으로 이루어진 긴 대화입니다. 여기서 몇 가지를 볼 수 있습니다.

Prompt caching: input token이 결국 prompt caching이 사용되는 지점까지 늘어났다는 것을 볼 수 있습니다(turn 6). 대화가 길어질수록 비용을 줄이고 속도를 높이는 데 도움이 됩니다!

다음 turn에는 10K context window 제한에 도달하여 compaction이 trigger됩니다.

In [8]:
response, usage = session.chat("Propose a title for the book")


🧹 Context window at 12847 tokens. Limit exceeded, compacting session memory...

------------------------------------------------------------
📝 New session memory created.
✅ Tokens reduced: 12,847 → 1526 (11,321 tokens saved, 88% reduction)
⏱️ Compaction time: 41.42s (user waiting...)
 Cache used: True
------------------------------------------------------------

User: Propose a title for the book

Assistant: 
Based on the story's core themes and imagery, here are my title proposals:

## Primary Recommendation

**The Cartographer's Daughter**

This works on multiple levels:
- Emma is metaphorically Amos Frost's "daughter" in mission—inheriting and completing his work
- Patricia (literal descendant of Frost's assistant) becomes Emma's accomplice
- Evokes the weight of inheritance, legacy, and what we pass down
- "Cartographer" immediately signals the map/truth theme
- Has literary gravitas appropriate for the story's tone

## Alternates

... (20 more lines)
Input=1,813, Prompt cached us

여기서 agent가 대화를 compact하는 데 **40초 넘게** 걸렸다는 것을 볼 수 있습니다. 기존 compaction을 사용했으므로 사용자는 Claude가 대화를 compact할 때까지 기다려야 하며, 이는 이상적인 사용자 경험이 아닙니다.

아래에서 compaction 결과를 볼 수 있습니다. 대화의 핵심 요소를 2K token 미만으로 담았습니다.

In [9]:
print(session.summary)

## User Intent
Create short story about young detective solving mysterious case in small town. Initially requested "3 well thought out plot ideas." Rejected first batch as not unique enough, requested "something more unique and unexpected." Accepted "The Cartographer's Grave" concept. Then requested: character backstory/motivations development, detailed chapter outline, and drafted chapters.

## Completed Work

**Approved Plot: "The Cartographer's Grave"**
- Ridgeway (pop. 3,200, mountain town) experiencing systematic address changes
- 12-year-old Emma Lancaster (terminal brain cancer) changing signs at night to match 1874 surveyor Amos Frost's original map
- Frost was "disgraced," replaced by Marcus Bellamy (founding family) in 1875 re-survey
- Real conspiracy: Bellamy survey deliberately shifted property lines 200-400 feet east to steal valuable land from Pequawket family (Native American), who had mineral rights + 15% revenue contract
- Emma found Frost's materials in grandmother's 

## 즉시 Compaction

**즉시 compaction**에서는 soft token threshold에 도달하면 session memory를 선제적으로 생성합니다. 

사용자가 compaction을 trigger하거나 hard limit에 도달하면 요약이 이미 준비되어 있으므로 사용자는 기다릴 필요가 없습니다.

결과: 즉시 compaction, 대기 없음.


SESSION MEMORY COMPACTION (즉시)
```
────────────────────────────────────
Turn 1 → Turn 2 → ... → Turn K → Turn K+1 → ... → Turn N → ..  → CONTEXT 가득 참!
                            │                         │            │
                (soft token threshold 도달:        (갱신          │
               session memory 초기화)          trigger)        │
                            │                                      │
                            │                         │            │
                            ▼                         ▼            │
                       ┌────────┐                ┌────────┐        │
                       │ 생성 │                │ 갱신 │        │
                       │ memory │ (백그라운드)   │ memory │        │
                       └────────┘                └────────┘        │
                            │                         │            │
                            ▼                         ▼            ▼
                     📝 session-memory.md ──────────────────► 즉시 교체!
                       (지속 갱신)
```

**갱신 trigger:** 첫 요약은 초기 soft token limit 이후 생성됩니다. 갱신은 이후 매 turn마다 또는 자연스러운 분기점에서 주기적으로(예: 약 10k token 또는 3회 이상 tool call마다) trigger할 수 있습니다.

이 `InstantCompactingChatSession` class는 백그라운드 실행에 **threading**을 사용합니다.
1. **`threading.Thread`** - blocking 없이 백그라운드에서 memory 갱신 실행
2. **Thread-safe 상태** - `threading.Lock`으로 공유 memory를 안전하게 갱신
3. **Daemon thread** - 백그라운드 작업이 프로그램 종료를 막지 않음
4. **즉시 compaction** - context가 가득 차면 미리 구축한 memory로 교체하기만 하면 됨

In [10]:
import threading
import time


class InstantCompactingChatSession:
    """
    점진적인 백그라운드 갱신으로 session memory를 유지합니다.

    핵심: 매 turn 후 백그라운드에서 memory를 갱신하므로,
    compaction이 필요할 때 요약이 이미 준비되어 즉시 교체할 수 있습니다!
    """

    def __init__(
        self,
        system_message="You are a helpful assistant",
        context_limit: int = 12000,
        min_tokens_to_init: int = 7500,
        min_tokens_between_updates: int = 2000,
    ):
        # Threshold
        self.context_limit = context_limit  # model 제한을 넘지 않도록 대화가 compact되는 지점
        self.min_tokens_to_init = min_tokens_to_init  # 초기 memory 생성을 trigger할 token 수; 기존 compaction과 달리 백그라운드에서 선제적으로 진행됨
        self.min_tokens_between_updates = min_tokens_between_updates  # memory 갱신을 trigger할 token 수. 초기 memory 생성 후 추가 compaction(memory 갱신)이 필요할 때만 적용

        # 대화 상태
        self.system_message = system_message
        self.messages = []
        self.current_context_window_tokens = 0

        # Session memory 상태
        self.session_memory = None  # session memory의 compact된 대화; 데모는 메모리에 저장하지만 production에서는 session_memory.md 파일에 기록
        self.last_summarized_index = (
            0  # session memory에 포함된 마지막 message의 index
        )
        self.tokens_at_last_update = 0  # 마지막 memory 갱신의 token을 추적하고 다음 갱신을 trigger할 만큼 새 token이 추가됐는지 확인

        # 백그라운드 갱신 추적
        self._update_thread: threading.Thread | None = None
        self.last_update_time = None
        self._lock = threading.Lock()

    def chat(self, user_message: str) -> tuple[str, anthropic.types.Usage, str | None]:
        """백그라운드 session memory 갱신을 사용해 chat turn을 처리합니다."""

        if self.current_context_window_tokens + estimate_tokens(user_message) >= self.context_limit:
            self.compact()  # trigger될 때 compaction은 이미 생성되었으며 즉시 교체하기만 함

        self.messages.append({"role": "user", "content": user_message})

        response = client.messages.create(
            model=MODEL,
            max_tokens=3500,
            system=self.system_message,
            messages=add_cache_control(self.messages),
        )

        assistant_message = response.content[0].text
        self.messages.append({"role": "assistant", "content": assistant_message})

        # cache를 포함한 token 사용량 계산
        cache_read = getattr(response.usage, "cache_read_input_tokens", 0) or 0
        total_input = response.usage.input_tokens + cache_read

        # context window token 갱신(cached token도 context에 포함되므로 함께 계산)
        self.current_context_window_tokens = total_input + response.usage.output_tokens

        # 핵심 차이: compaction이 필요해지기 전에 필요에 따라 백그라운드 memory 갱신을 선제적으로 trigger
        background_status = None
        if self._should_init_memory() or self._should_update_memory():
            self._trigger_background_update()
            background_status = "initializing" if self.session_memory is None else "updating"

        # cache 통계가 포함된 사용량 정보 반환
        return assistant_message, response.usage, background_status

    # session memory 초기화 시점을 결정하는 helper method
    def _should_init_memory(self) -> bool:
        return (
            self.session_memory is None
            and self.current_context_window_tokens >= self.min_tokens_to_init
        )

    # memory 갱신 여부를 결정하는 helper method
    def _should_update_memory(self) -> bool:
        if self.session_memory is None:
            return False
        tokens_since = self.current_context_window_tokens - self.tokens_at_last_update
        return tokens_since >= self.min_tokens_between_updates

    # 초기 session memory 생성 method
    def _create_session_memory(self, messages: list[dict]) -> str:
        """message들에서 초기 session memory를 생성합니다."""
        # 주 대화와 cache를 공유하도록 user message에 compaction 지시 넣기
        compaction_messages = [{"role": "user", "content": SESSION_MEMORY_PROMPT}]
        response = client.messages.create(
            model=MODEL,
            max_tokens=5000,
            system=self.system_message,  # cache 공유를 위해 주 대화와 동일
            messages=add_cache_control(messages) + compaction_messages,
        )
        summary, _ = remove_thinking_blocks(
            response.content[0].text
        )  # session memory에는 필요하지 않으므로 <think> block 제거
        print(
            f"   [Background] Initial session memory created. Cache hit={getattr(response.usage, 'cache_read_input_tokens', 0) > 0}"
        )
        return summary

    def _update_session_memory(self, new_messages: list[dict]) -> str:
        """새 message로 기존 session memory를 갱신합니다. 실제로는 전체 재생성 대신 파일 편집으로 할 수 있습니다. 하지만 여기서는 데모를 위해 전체 재생성합니다."""
        # 주 대화와 cache를 공유하도록 user message에 compaction 지시 넣기
        compaction_update_messages = [
            {
                "role": "user",
                "content": SESSION_MEMORY_PROMPT
                + f"""There is an existing session memory: {self.session_memory}. Return the entire session memory with updates to reflect new messages.""",
            }
        ]
        response = client.messages.create(
            model=MODEL,
            max_tokens=5000,
            system=self.system_message,
            messages=new_messages
            + compaction_update_messages,  # 대신 prompt caching을 사용하려면 여기서 add_cache_control(self.messages) 사용
        )
        updated_summary, _ = remove_thinking_blocks(
            response.content[0].text
        )  # session memory에는 필요하지 않으므로 <think> block 제거
        print("   [Background] Session memory updated.")
        return updated_summary

    # 백그라운드 memory 갱신 method
    def _background_memory_update(
        self, messages_snapshot: list[dict], snapshot_index: int, current_tokens: int
    ) -> None:
        """background thread에서 session memory 갱신을 실행합니다."""
        try:
            with self._lock:
                current_session_memory = self.session_memory
                last_index = self.last_summarized_index

            if current_session_memory is None:
                new_memory = self._create_session_memory(messages_snapshot)
            else:
                # 마지막 요약 이후의 새 message 얻기
                new_messages = messages_snapshot[last_index:]
                if not new_messages:
                    return
                new_memory = self._update_session_memory(new_messages)

            # 상태 갱신(thread-safe)
            with self._lock:
                self.session_memory = new_memory
                self.last_summarized_index = snapshot_index
                self.tokens_at_last_update = current_tokens
                self.last_update_time = time.time()

        except Exception as e:
            print(f"   [Background] Error updating memory: {e}")

    # 백그라운드 갱신이 한 번에 하나만 실행되게 함. 이미 실행 중이면 다른 갱신 시작을 건너뛰고, 아니면 새 thread에서 갱신 시작.
    def _trigger_background_update(self):
        """백그라운드 session memory 갱신을 trigger합니다."""
        if self._update_thread is not None and self._update_thread.is_alive():
            return

        messages_snapshot = self.messages.copy()
        snapshot_index = len(messages_snapshot)
        current_tokens = self.current_context_window_tokens

        self._update_thread = threading.Thread(
            target=self._background_memory_update,
            args=(messages_snapshot, snapshot_index, current_tokens),
            daemon=True,
        )
        self._update_thread.start()

    # compact 함수
    def compact(self) -> None:
        """미리 구축한 session memory를 사용한 즉시 compaction."""
        prev_msg_count = len(self.messages)

        # session memory 준비 상태 확인. 보통 문제없지만 안전을 위해 추가.
        if self.session_memory is None:
            if self._update_thread is not None and self._update_thread.is_alive():
                print("   ⏳ Waiting for background memory update...")
                self._update_thread.join(timeout=30.0)

            if self.session_memory is None:
                print("   ⚠️  No pre-built memory, creating synchronously...")
                start = time.perf_counter()
                self.session_memory = self._create_session_memory(self.messages)
                elapsed = time.perf_counter() - start
                print(f"   ⏱️  Took {elapsed:.2f}s (but should be instant normally!)")
                self.last_summarized_index = len(self.messages)

        with self._lock:
            unsummarized = self.messages[self.last_summarized_index :]
            summary_message = [
                {
                    "role": "user",
                    "content": f"""This session is being continued from a previous conversation. Here is the session memory: {self.session_memory}.Continue from where we left off.""",
                }
            ]
            self.messages = summary_message + unsummarized
            self.last_summarized_index = 1

            print(f"\n{'=' * 60}")
            print(f"⚡ INSTANT COMPACTION! Messages: {prev_msg_count} → {len(self.messages)}")
            print("   Session memory was pre-built (no wait time!)")
            print(f"{'=' * 60}")

/root/.pyenv/versions/3.13.11/lib/python3.13/site-packages/coconut/compiler/util.py:403: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  grammar.streamline()
/root/.pyenv/versions/3.13.11/lib/python3.13/site-packages/coconut/compiler/util.py:457: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  result = add_action(grammar, unpack).parseWithTabs().transformString(text)


### 즉시 Compaction 사용 예제

In [12]:
# 데모용 낮은 threshold - production에서는 더 높은 값 사용
session = InstantCompactingChatSession(
    system_message=SYSTEM_PROMPT,
)

messages = [
    "I want to create a story about a young detective solving a mysterious case in a small town. Generate 3 well thought out plot ideas for me to consider.",
    "I don't like those ideas, can you think of one plot something more unique and unexpected?",
    "Ok I like it. Can you help me develop the main character's backstory and motivations?",
    "Can you draft a detailed outline for the story, breaking it down into chapters and key events?",
    "Can you draft me a first chapter based on the plot and character ideas we've discussed so far? Make it around 2,000 words.",
    "Can you draft a second chapter that builds on the first one?",
]
print("Starting conversation with instant compacting chat session...\n")

turn_count = 0
for message in messages:
    response, usage, background_status = session.chat(message)
    turn_count += 1

    # cache 통계 계산
    cache_read = getattr(usage, "cache_read_input_tokens", 0) or 0
    cache_created = getattr(usage, "cache_creation_input_tokens", 0) or 0
    total_input = usage.input_tokens + cache_read

    print(f"{'=' * 60}")
    print(f"Turn {turn_count}:")
    print(f"\nUser: {message}")
    print(f"\nAssistant: \n{truncate_response(response, max_lines=3)}")
    print("\nToken Usage:")
    print(f"  Input: {total_input:,} (new: {usage.input_tokens:,}, cached: {cache_read:,})")
    print(f"  Output: {usage.output_tokens:,}")
    print(
        f"  Messages: {len(session.messages)} | Memory: {'ready' if session.session_memory else 'not yet'}"
    )

    if cache_read > 0:
        cache_pct = (cache_read / total_input) * 100
        print(f"  ✓ Cache hit! {cache_pct:.0f}% of input from cache")

    if background_status:
        print(f"\n  [Background] Proactively {background_status} session memory...")
        print(f"  Context window: {session.current_context_window_tokens:,} tokens")

    print()

Starting conversation with instant compacting chat session...

Turn 1:

User: I want to create a story about a young detective solving a mysterious case in a small town. Generate 3 well thought out plot ideas for me to consider.

Assistant: 
# Three Mystery Plot Ideas

## 1. **The Vanishing Choir**
... (36 more lines)

Token Usage:
  Input: 317 (new: 317, cached: 0)
  Output: 902
  Messages: 2 | Memory: not yet

Turn 2:

User: I don't like those ideas, can you think of one plot something more unique and unexpected?

Assistant: 
# **The Forgetting House**

**Setup:** Your young detective arrives in Ember Falls to investigate a string of burglaries—except the victims don't realize they've been robbed until weeks later. A woman discovers her wedding ring gone and insists she lost it yesterday, but security footage shows she hasn't worn it in a month. A man reports his grandfather's watch stolen, then his sister shows him photos proving he sold it himself at a pawn shop—which he has no mem

In [13]:
message = "What did we just talk about? Give me one sentence"
response, usage, background_status = session.chat(message)

# cache 통계 계산
cache_read = getattr(usage, "cache_read_input_tokens", 0) or 0
total_input = usage.input_tokens + cache_read

print(f"\nUser: {message}")
print(f"\nAssistant: \n{truncate_response(response, max_lines=3)}")
print("\nToken Usage:")
print(f"  Input: {total_input:,} (new: {usage.input_tokens:,}, cached: {cache_read:,})")
print(f"  Output: {usage.output_tokens:,}")
print(
    f"  Messages: {len(session.messages)} | Memory: {'ready' if session.session_memory else 'not yet'}"
)

if cache_read > 0:
    cache_pct = (cache_read / total_input) * 100
    print(f"  ✓ Cache hit! {cache_pct:.0f}% of input from cache")


⚡ INSTANT COMPACTION! Messages: 12 → 3
   Session memory was pre-built (no wait time!)

User: What did we just talk about? Give me one sentence

Assistant: 
I drafted Chapter 2 where Casey arrives at Rosemont Manor, interviews Iris (who deflects questions about her past and shows moments of disorientation), and realizes through comparing photos that Iris Hale is definitely their missing grandmother Iris Whitmore.

Token Usage:
  Input: 5,490 (new: 5,490, cached: 0)
  Output: 60
  Messages: 5 | Memory: ready


여기서는 context 제한에 도달하자마자 session memory가 즉시 교체되어, 사용자가 응답을 기다리는 시간이 전혀 없었다는 것을 볼 수 있습니다!

## 고급: Prompt Caching 이해


백그라운드 갱신은 prompt caching으로 **약 10배 저렴하게** 만들 수 있습니다. 요령은 다음과 같습니다.
1. 백그라운드 요약기에 **전체 대화** 전달
2. 후속 요청이 cache hit되도록 `cache_control` marker 추가
3. 새로운 "이것을 요약하라" 지시만 정상 가격으로 청구

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│                    긴 대화를 위한 PROMPT CACHING                        │
├─────────────────────────────────────────────────────────────────────────────────┤
│                                                                                 │
│  CACHING 미사용: 매 turn마다 전체 context의 정상 가격 지불                 │
│  ════════════════════════════════════════════════════════════                   │
│                                                                                 │
│  Turn 1:  [System][User1][Asst1]                         →  500 tokens  @ $3/M │
│  Turn 2:  [System][User1][Asst1][User2][Asst2]           → 1500 tokens  @ $3/M │
│  Turn 3:  [System][User1][Asst1][User2][Asst2][User3]... → 3000 tokens  @ $3/M │
│  Turn 4:  [System][User1][Asst1][User2][Asst2][User3]... → 5000 tokens  @ $3/M │
│           ─────────────────────────────────────────────                         │
│                                              합계: 10,000 tokens = $0.030      │
│                                                                                 │
│                                                                                 │
│  CACHING 사용: 정상 가격은 한 번, 이후 prefix는 90% 할인                │
│  ═══════════════════════════════════════════════════════════════                │
│                                                                                 │
│  Turn 1:  [System][User1][Asst1]◆                        →  500 tokens  @ $3/M │
│                                ▲                            (cache 생성)    │
│                          cache breakpoint                                       │
│                                                                                 │
│  Turn 2:  [System][User1][Asst1][User2][Asst2]◆                                │
│           ╰─────── cached ──────╯                                              │
│                500 @ $0.30/M + 1000 new @ $3/M  =  $0.0032                     │
│                                                                                 │
│  Turn 3:  [System][User1][Asst1][User2][Asst2][User3][Asst3]◆                  │
│           ╰──────────── cached ─────────────╯                                  │
│               1500 @ $0.30/M + 1500 new @ $3/M  =  $0.0050                     │
│                                                                                 │
│  Turn 4:  [System][User1][Asst1][User2][Asst2][User3][Asst3][User4][Asst4]◆    │
│           ╰───────────────────── cached ─────────────────────╯                 │
│                     3000 @ $0.30/M + 2000 new @ $3/M  =  $0.0069               │
│           ─────────────────────────────────────────────                         │
│                                              합계: $0.0166  (45% 절약)     │
│                                                                                 │
├─────────────────────────────────────────────────────────────────────────────────┤
│                                                                                 │
│  COMPACTION + CACHING: 이중 이점                                           │
│  ════════════════════════════════════                                           │
│                                                                                 │
│    주 대화                      백그라운드 요약기                         │
│    ─────────                      ─────────────────────                         │
│                                                                                 │
│  [대화 증가...]          [동일한 대화 prefix]◆ + [요약!]   │
│         │                                    │                                  │
│         │                         Cache hit! 요약 프롬프트                      │
│         │                         비용만 지불                      │
│         │                                    │                                  │
│         ▼                                    ▼                                  │
│  Context 제한 도달  ──────►  Session memory 즉시 준비                │
│                                  (백그라운드에서 저렴하게 구축)                  │
│                                                                                 │
│  ┌──────────────────────────────────────────────────────────────────────────┐  │
│  │  핵심: 백그라운드 요약기는 방금 주 대화에 보낸 동일한 대화     │  │
│  │  prefix를 재사용하므로 자동 cache hit!       │  │
│  └──────────────────────────────────────────────────────────────────────────┘  │
│                                                                                 │
└─────────────────────────────────────────────────────────────────────────────────┘

◆ = cache_control breakpoint (이 지점 이전의 모든 내용 cache)
```

### Compaction에서 이것이 중요한 이유

| Scenario | 백그라운드 갱신당 비용 | 참고 |
|----------|---------------------------|-------|
| Caching 없음 | 전체 input 비용 | 5,000 token × $3/M = $0.015 |
| Caching 있음 | input 비용의 약 10% | 새 500 + cached 4,500 = $0.003 |
| **절감** | **약 80%** | 여러 갱신에서 누적됨 |

대화가 길수록 절감 효과가 커집니다. 바로 compaction이 가장 필요한 때입니다!

### Caching 작동 방식

핵심은 `_add_cache_control()`과 `_create_session_memory_cached()`에 있습니다.

```python
# 1. 마지막 대화 message에 cache_control 표시
{
    "role": "user",
    "content": [{
        "type": "text",
        "text": msg["content"],
        "cache_control": {"type": "ephemeral"}  # <-- cache breakpoint 생성
    }]
}

# 2. system prompt도 표시
system=[{
    "type": "text",
    "text": "You are a session memory agent...",
    "cache_control": {"type": "ephemeral"}
}]
```

**작동하는 이유:**
- 첫 백그라운드 갱신에서 `[System + Messages]`의 cache entry 생성
- 같은 message prefix를 사용하는 후속 갱신에서 **cache hit**
- 새 요약 지시만 정상 가격으로 청구
- Cache entry의 TTL은 5분이므로 빠른 갱신에서 이점이 가장 큼

**비용 계산:**
- Caching 없음: 5,000 token × $3.00/1M = 갱신당 $0.015
- Caching 있음: 새 500 token × $3.00/1M + cached 4,500 × $0.30/1M = $0.00285
- 백그라운드 요약 비용 **약 80% 절감**

## 결론

이 cookbook에서는 session memory compaction을 통해 장기 Claude 대화를 관리하는 방법을 배웠습니다.

### 다룬 내용

✅ **효과적인 compaction 프롬프트** - 불필요한 내용은 버리고 사용자 의도, 완료한 작업, 오류, 진행 중인 작업, 핵심 reference를 보존하도록 session memory 구성

✅ **즉시 compaction** - background threading으로 session memory를 선제적으로 구축하여 context 제한에 도달했을 때 사용자 대기 시간 제거

✅ **비용 절감을 위한 Prompt caching** - 대화 prefix cache를 재사용해 백그라운드 갱신 비용 약 80% 절감

✅ **기존 방식 대 즉시 방식 패턴** - application 요구에 따라 각 접근 방식을 언제 사용할지 이해

### 핵심 요점

1. **최신성에 높은 가중치를 두세요** - 대화의 끝부분이 현재 작업 context입니다.
2. **사용자의 수정 사항을 원문 그대로 보존하세요** - model이 이전 행동으로 되돌아가는 것을 막습니다.
3. **memory를 선제적으로 구축하세요** - context 제한을 기다리지 말고 백그라운드 갱신을 일찍 시작하세요.
4. **prompt caching을 활용하세요** - 백그라운드 요약이 주 대화와 cache를 공유할 수 있습니다.

### 다음 단계

- **agentic workflow**: tool use를 사용하는 SDK 기반 자동 compaction은 [Automatic Context Compaction](../tool_use/automatic-context-compaction.ipynb) 참조
- **production**: session memory를 메모리에 유지하는 대신 disk에 저장하는 것 고려
- **최적화**: 비용과 최신성 사이의 균형을 위해 갱신 빈도 threshold 실험